In [1]:
INPUT_WB_NAME  = "2026 Inputs for Apps.xlsx"

####
INPUT_WS_NAME  = "FUTURES INPUTS"
INPUT_TBL_NAME = "FUTURES_INPUTS"
####

In [2]:
# --- system setup ---
import sys
import os
sys.path.append(os.path.abspath(".."))

# --- autoreload ---
%load_ext autoreload
%autoreload 2

In [3]:
import asyncio
from collections import defaultdict
import numpy as np

In [4]:
# --- builders ---
from fin_insts import make_single_leg_fin_insts, FutureSpread #, BestOf, Synthetic

In [5]:
# --- IBKR ---
from ibkr.Class_IBKR_IB import IBKR_IB
# from ibkr.Class_IBKR_TWS import IBKR_TWS
ibkr = IBKR_IB()

In [6]:
# --- feeds ---
from ws_feeds import WSFeedManager

In [7]:
# --- utils ---
# from other.Graph_Theory import find_all_node_permutations, connect_nodes_with_edges
from output.Output_Methods import create_output
from output.Class_xlWings import xlWings
xlw = xlWings()

In [8]:
# --- trading strategy ---
# from strategies import Strategy, TradePackage

In [9]:
# CONSTANTS

DB_WB_NAME  = "2026 Crypto Products Database.xlsx"

OUTPUT_COLS = [
               'my_prod_type',
               'my_fi_name',
               'my_pf_name',
    
               'numerator_currency',
               'denominator_currency',
                              
               'price_mkt_bid',
               'price_mkt_ask',
    
               'scalar_price_mkt_to_unit',
    
               'price_unit_bid',
               'price_unit_ask'
        ]

In [10]:
async def standard_startup(xlw, INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME):

    df = xlw.get_df(INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME, table=True)
    input_dict = df.set_index('Keys')['Values'].to_dict()
    
    wb  = input_dict['input workbook name']
    ws  = input_dict['true/false sheet name']
    tbl = input_dict['true/false table name']
    true_false_df = xlw.get_df(wb, ws, tbl, table=True)
    
    if 'TRUE/FALSE' not in true_false_df.columns:
        true_false_df = true_false_df.set_index('Keys').T

    true_false_df = true_false_df[true_false_df['TRUE/FALSE'] == True]
    
    wb  = DB_WB_NAME
####    
    ws  = input_dict['crypto long name']
    tbl = input_dict['crypto abbrev'] + "_static_data_table"
####
    
    db_df = xlw.get_df(wb, ws, tbl, table=True)
    
    merged_df = true_false_df.merge(db_df,how='left',on=['my_fi_name', 'my_pf_name'])

    fin_inst_objs_list = make_single_leg_fin_insts(merged_df)

    return input_dict, fin_inst_objs_list

In [11]:
async def main():

    input_dict, objs_list = await standard_startup(xlw, INPUT_WB_NAME, INPUT_WS_NAME, INPUT_TBL_NAME)

    ws_objs_list = [obj for obj in objs_list if obj.my_pf_name != 'IBKR']
    ws_feed      = WSFeedManager(ws_objs_list)

    await ws_feed.complete_fi_objects()   
        
    ibkr_objs_list     = [obj for obj in objs_list if obj.my_pf_name == 'IBKR']
    if ibkr_objs_list:
        await ibkr.connect()
        print("IBKR connected:", ibkr.ib.isConnected())
        
        await asyncio.gather(*(ibkr.create_simple_contract(obj) for obj in ibkr_objs_list))
        await asyncio.gather(*(ibkr.complete_obj(obj) for obj in ibkr_objs_list))
    
    #''' 
    # insert ibkr BAG instruments here 
    futures_list = [obj for obj in ibkr_objs_list if obj.my_prod_type == 'future']
    bag_objs_list = FutureSpread.make_spreads(futures_list)
    await asyncio.gather(*(ibkr.create_bag_contract(obj) for obj in bag_objs_list))        
    #'''                 
        
    '''   
    # insert synthetic instruments here 
    syn_objs_list = Synthetic.make_syn_futures_list(futures_list, bag_objs_list)
    ''' 

    ''' 
    # insert bestOf instruments here
    bo_objs_dict = defaultdict(list)
    for obj in futures_list:
        bo_objs_dict[obj.my_fi_name].append(obj)
    for obj in syn_objs_list:
        bo_objs_dict[obj.final_contract].append(obj)
    
    bo_objs_list = []
    attr_list = [('price_unit_bid', max), 
                 ('price_unit_ask', min)]
    for my_name, obj_list in bo_objs_dict.items():
        bo_obj = BestOf(my_name, obj_list, attr_list)
        bo_objs_list.append(bo_obj)
    
    bo_objs_list.sort(key=lambda obj: obj.my_fi_name)
    '''
    
    output_list = [
        #*bo_objs_list,
        *ws_objs_list,
        *ibkr_objs_list,
        *bag_objs_list,
        #*syn_objs_list,
        #*strat_objs_list
                ]
    
    # Run all streams concurrently
    tasks = []
    tasks.append(asyncio.create_task(ws_feed.run()))
    tasks.append(asyncio.create_task(create_output(input_dict, output_list, OUTPUT_COLS)))
    if ibkr_objs_list:
        tasks.append(asyncio.create_task(ibkr.start_streams(ibkr_objs_list)))
        tasks.append(asyncio.create_task(ibkr.start_streams(bag_objs_list)))

    '''
    for obj in bo_objs_list:
        tasks.append(asyncio.create_task(obj.run_timer())) 
    
    await asyncio.sleep(15)


    await strat.done_event.wait()
    
    # then cancel everything else
    for task in tasks:
        task.cancel()
    
    # optional: wait for clean cancellation
    await asyncio.gather(*tasks, return_exceptions=True)
    
    # disconnect IBKR
    ibkr.ib.disconnect()
    
    print("Program finished cleanly.")
    '''

In [12]:
await main()

IBKR connected: True
1 Ticker(contract=Contract(secType='FUT', conId=835061447, symbol='BRR', lastTradeDateOrContractMonth='20260529', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCK6', tradingClass='BTC')) 

1 Ticker(contract=Contract(secType='FUT', conId=751356962, symbol='BRR', lastTradeDateOrContractMonth='20260626', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCM6', tradingClass='BTC')) 

1 Ticker(contract=Contract(secType='FUT', conId=850790355, symbol='BRR', lastTradeDateOrContractMonth='20260731', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCN6', tradingClass='BTC')) 

1 Ticker(contract=Contract(secType='FUT', conId=859040542, symbol='BRR', lastTradeDateOrContractMonth='20260828', multiplier='5', exchange='CME', currency='USD', localSymbol='BTCQ6', tradingClass='BTC')) 

1 Ticker(contract=Contract(secType='FUT', conId=772435574, symbol='BRR', lastTradeDateOrContractMonth='20260925', multiplier='5', exchange='CME', currency=

2026-05-21_13-03-40
2026-05-21_13-03-50
2026-05-21_13-04-00
2026-05-21_13-04-11
2026-05-21_13-04-21
2026-05-21_13-04-31
2026-05-21_13-04-41
2026-05-21_13-04-51
2026-05-21_13-05-01
2026-05-21_13-05-11
